# insurance-distill

**Distil a GBM into multiplicative GLM factor tables that a rating engine can actually consume.**

Your CatBoost model outperforms the GLM. Your rating engine (Radar, Emblem, or any multiplicative system) cannot load a gradient boosted tree. This library bridges that gap: fit a Poisson GLM on the GBM's pseudo-predictions, bin continuous variables optimally, and export factor tables with Gini retention and segment deviation diagnostics.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/burning-cost/insurance-distill/blob/main/notebooks/quickstart.ipynb)

In [ ]:
!pip install -q "insurance-distill[catboost]" polars numpy scikit-learn

## Synthetic motor data and a fitted CatBoost model

We generate 5,000 synthetic UK motor policies with a Poisson claim count target, then fit a CatBoost Poisson model. This represents the "GBM sitting on a server" that the distillation will translate into a factor table.

In [ ]:
import numpy as np
import polars as pl
from catboost import CatBoostRegressor, Pool

rng = np.random.default_rng(42)
n = 5_000

driver_age   = rng.integers(17, 75, n).astype(float)
vehicle_age  = rng.integers(0, 15, n).astype(float)
ncd_years    = rng.integers(0, 9, n).astype(float)
area         = rng.integers(0, 6, n).astype(float)  # 0-5 area bands
annual_miles = rng.integers(3000, 20000, n).astype(float)
exposure     = rng.uniform(0.3, 1.0, n)

log_rate = (
    -2.5
    + 0.40 * (driver_age < 25).astype(float)   # young driver load
    - 0.10 * ncd_years
    + 0.25 * (vehicle_age > 10).astype(float)   # old vehicle load
    + 0.08 * area
)
y = rng.poisson(np.exp(log_rate) * exposure)

X = pl.DataFrame({
    "driver_age":   driver_age.tolist(),
    "vehicle_age":  vehicle_age.tolist(),
    "ncd_years":    ncd_years.tolist(),
    "area":         area.tolist(),
    "annual_miles": annual_miles.tolist(),
})

n_train = int(0.8 * n)
X_train = X[:n_train]
y_train = y[:n_train]
exp_train = exposure[:n_train]

# Fit CatBoost Poisson
pool = Pool(data=X_train.to_pandas(), label=y_train, weight=exp_train)
cb = CatBoostRegressor(
    loss_function="Poisson",
    iterations=300,
    depth=5,
    learning_rate=0.05,
    verbose=0,
    random_seed=42,
)
cb.fit(pool)
print(f"GBM fitted on {n_train:,} training policies.")
print(f"Feature importances: {dict(zip(X.columns, cb.get_feature_importance().round(1)))}")

## Distil into a surrogate GLM

`SurrogateGLM` fits a Poisson GLM on the GBM's pseudo-predictions (not on actual claims). This removes the noise of individual claim events — the GBM has already smoothed that out. The result is a multiplicative factor table that approximates the GBM's predictions in the format a rating engine expects.

In [ ]:
from insurance_distill import SurrogateGLM

surrogate = SurrogateGLM(
    model=cb,
    X_train=X_train,
    y_train=y_train,
    exposure=exp_train,
    family="poisson",
)
surrogate.fit(max_bins=8)
print("Surrogate GLM fitted.")

## Validation: how much does the surrogate retain?

The key diagnostic is Gini ratio — what fraction of the GBM's discriminatory power does the GLM surrogate retain? Above 90% is generally acceptable; above 95% is excellent.

In [ ]:
report = surrogate.report()
print(report.metrics.summary())

## Inspect factor tables

Each factor table is a Polars DataFrame with `level`, `log_coefficient`, and `relativity` columns. The base level always has `relativity = 1.0`. All other levels are multiplicative factors relative to the base.

In [ ]:
# NCD factor table — should be monotone decreasing
print("NCD years factor table:")
print(surrogate.factor_table("ncd_years"))

print()
print("Driver age factor table (binned):")
print(surrogate.factor_table("driver_age"))

## What you should see

- Gini ratio: typically 90–97% on this DGP. The GLM surrogate retains most of the GBM's discrimination.
- NCD: monotone decreasing relativities — 0 NCD = highest multiplier, 8+ NCD = lowest.
- Driver age: a load for the youngest band (17–24) reflecting the young driver effect in the DGP.
- Max segment deviation: ideally under 8%. This tells you the worst cell where the GLM under/over-prices relative to the GBM.

The output format maps directly to rating engine factor table CSV imports. `surrogate.export_csv("factors/")` writes one CSV per variable.

## Next steps

- **`binning_method="isotonic"`** — enforce monotone NCD and mileage factors
- **`interaction_pairs=[("driver_age", "area")]`** — add two-way interaction terms
- **`surrogate.export_csv("output/", prefix="motor_freq_")`** — export for Radar/Emblem import

**GitHub:** https://github.com/burning-cost/insurance-distill  
**PyPI:** https://pypi.org/project/insurance-distill/